In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("NYC_TAXI").getOrCreate()
df_nytaxi=spark.read.format("delta").load("/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/")

In [0]:
df_nytaxi.show(truncate=False)

+---------+-------------------+-------------------+---------------+-------------+----------------+---------------+------------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+------------+
|vendor_id|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|pickup_longitude|pickup_latitude|rate_code_id|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|total_amount|
+---------+-------------------+-------------------+---------------+-------------+----------------+---------------+------------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+------------+
|VTS      |2009-11-07 19:44:00|2009-11-07 19:49:00|2              |0.74         |-73.992127      |40.734658      |NULL        |NULL              |-73.99197        |40.729115       |CASH        |4.5        |0.0  |0.5    |0.0   

In [0]:
df_nytaxi.count()

1611611035

In [0]:
df_nytaxi.schema.names

['vendor_id',
 'pickup_datetime',
 'dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'pickup_longitude',
 'pickup_latitude',
 'rate_code_id',
 'store_and_fwd_flag',
 'dropoff_longitude',
 'dropoff_latitude',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'total_amount']

In [0]:
display(df_nytaxi.limit(5))

vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,rate_code_id,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,total_amount
VTS,2009-11-07T19:44:00.000Z,2009-11-07T19:49:00.000Z,2,0.74,-73.992127,40.734658,null,null,-73.99197,40.729115,CASH,4.5,0.0,0.5,0.0,0.0,5.0
VTS,2009-11-08T02:06:00.000Z,2009-11-08T02:12:00.000Z,1,1.04,-74.008553,40.719682,null,null,-74.01615,40.709963,CASH,5.7,0.5,0.5,0.0,0.0,6.7
VTS,2009-11-08T03:57:00.000Z,2009-11-08T04:09:00.000Z,1,4.05,-74.007742,40.717097,null,null,-73.986758,40.768467,CASH,11.7,0.5,0.5,0.0,0.0,12.7
VTS,2009-11-05T12:56:00.000Z,2009-11-05T12:58:00.000Z,1,0.33,-73.992995,40.71258,null,null,-73.992995,40.71258,CASH,3.3,0.0,0.5,0.0,0.0,3.8
VTS,2009-11-05T08:35:00.000Z,2009-11-05T08:41:00.000Z,1,0.6,-74.00921,40.712933,null,null,-74.002663,40.716882,CASH,4.5,0.0,0.5,0.0,0.0,5.0


Checking max amount of fare

In [0]:
from pyspark.sql.functions import max
df_nytaxi.select(max("total_amount")).show()

+-----------------+
|max(total_amount)|
+-----------------+
|         685908.1|
+-----------------+



In [0]:
from pyspark.sql.functions import max,col
max_taxi_fare=df_nytaxi.select(max("total_amount")).collect()[0][0]

In [0]:
display(df_nytaxi.filter(col("total_amount")==max_taxi_fare))

vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,rate_code_id,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,total_amount
VTS,2013-08-14T21:29:00.000Z,2013-08-14T21:53:00.000Z,1,11.62,-73.993377,40.764467,1,null,-73.877792,40.826705,CSH,33.0,0.5,0.5,0.0,0.0,685908.1


In [0]:
df_nytaxi.select("payment_type").distinct().show()

+------------------+
|      payment_type|
+------------------+
|               No |
|              CASH|
|40.773575000000001|
|                 0|
|40.747324999999996|
|               CAS|
|40.754798999999998|
|40.807485999999997|
|           Dispute|
|40.765663000000004|
|               NOC|
|40.850890999999997|
|40.772900999999997|
|40.689061000000002|
|40.752012000000001|
|40.789335000000001|
|40.820529000000001|
|40.769525999999999|
|40.775103999999999|
|40.749665999999998|
+------------------+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import count,lit
df_nytaxi.groupBy("payment_type").agg(count(lit(" ")).alias("payment_count")).filter(col("payment_count") > 2).show()

+------------------+-------------+
|      payment_type|payment_count|
+------------------+-------------+
|               No |       200451|
|              CASH|     69117503|
|                 0|    178291680|
|               CAS|     30792006|
|40.768954999999998|            3|
|                 2|     76841365|
|           Dispute|        94784|
|               NOC|      1195881|
|               5.5|        12225|
|40.754987999999997|            3|
|               Cas|     26053917|
|               CSH|    389969596|
|               CRD|    382212709|
|40.775128000000002|            3|
|                 4|       263962|
|               DIS|       365825|
|              Cash|     56282593|
|                 5|          114|
|               UNK|      1070012|
|              2.75|       221458|
+------------------+-------------+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import rlike
df_nytaxi.filter(col("payment_type").rlike("^[a-zA-Z]")).groupBy("payment_type").agg(count(lit(" ")).alias("payment_count")).filter(col("payment_count") > 2).show()

+------------+-------------+
|payment_type|payment_count|
+------------+-------------+
|         No |       200451|
|        CASH|     69117503|
|         CAS|     30792006|
|     Dispute|        94784|
|         NOC|      1195881|
|         Cas|     26053917|
|         CSH|    389969596|
|         CRD|    382212709|
|         DIS|       365825|
|        Cash|     56282593|
|         UNK|      1070012|
|      CREDIT|      2330599|
|   No Charge|       509194|
|      Credit|     42561382|
|         NA |        39986|
|         CRE|      3369965|
|         Dis|        43596|
|         Cre|     27416052|
+------------+-------------+



In [0]:
df_nytaxi.filter(col("payment_type").rlike("^[a-zA-Z]")).groupBy("payment_type").agg(count(lit(" ")).alias("payment_count")).filter(col("payment_count") > 2).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   ColumnarToRow
   +- PhotonResultStage
      +- PhotonFilter (payment_count#13649L > 2)
         +- PhotonGroupingAgg(keys=[payment_type#13218], functions=[finalmerge_count(merge count#13660L) AS count(1)#13650L])
            +- PhotonShuffleExchangeSource
               +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#10264]
                  +- PhotonShuffleExchangeSink hashpartitioning(payment_type#13218, 1024)
                     +- PhotonGroupingAgg(keys=[payment_type#13218], functions=[partial_count(1) AS count#13660L])
                        +- PhotonScan parquet [payment_type#13218] DataFilters: [isnotnull(payment_type#13218), RLIKE(payment_type#13218, ^[a-zA-Z])], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow], OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<payment_type:string>, Requir

In [0]:
df_nytaxi.explain()

== Physical Plan ==
*(1) ColumnarToRow
+- PhotonResultStage
   +- PhotonScan parquet [vendor_id#13207,pickup_datetime#13208,dropoff_datetime#13209,passenger_count#13210,trip_distance#13211,pickup_longitude#13212,pickup_latitude#13213,rate_code_id#13214,store_and_fwd_flag#13215,dropoff_longitude#13216,dropoff_latitude#13217,payment_type#13218,fare_amount#13219,extra#13220,mta_tax#13221,tip_amount#13222,tolls_amount#13223,total_amount#13224] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow], OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<vendor_id:string,pickup_datetime:timestamp,dropoff_datetime:timestamp,passenger_count:int,..., RequiredDataFilters: []


== Photon Explanation ==
The query is fully supported by Photon.


In [0]:
data_map=[('CAS','CASH'),('Cas','CASH'),('CSH','CASH'),('Cash','CASH')]
schema=['OLD_VALUE','New_value']
df_new=spark.createDataFrame(data_map,schema)
df_new.show()

+---------+---------+
|OLD_VALUE|New_value|
+---------+---------+
|      CAS|     CASH|
|      Cas|     CASH|
|      CSH|     CASH|
|     Cash|     CASH|
+---------+---------+



In [0]:
df_nytaxi.join(df_new,on=df_nytaxi.payment_type==df_new.OLD_VALUE,how="inner").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   ColumnarToRow
   +- PhotonResultStage
      +- PhotonBroadcastHashJoin [payment_type#13218], [OLD_VALUE#13708], Inner, BuildRight, false, true
         :- PhotonScan parquet [vendor_id#13207,pickup_datetime#13208,dropoff_datetime#13209,passenger_count#13210,trip_distance#13211,pickup_longitude#13212,pickup_latitude#13213,rate_code_id#13214,store_and_fwd_flag#13215,dropoff_longitude#13216,dropoff_latitude#13217,payment_type#13218,fare_amount#13219,extra#13220,mta_tax#13221,tip_amount#13222,tolls_amount#13223,total_amount#13224] DataFilters: [isnotnull(payment_type#13218)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow], OptionalDataFilters: [hashedrelationcontains(payment_type#13218)], PartitionFilters: [], ReadSchema: struct<vendor_id:string,pickup_datetime:timestamp,dropoff_datetime:timestamp,passenger_count:int,..

In [0]:
df_nytaxi.join(df_new.hint("MERGE"),on=df_nytaxi.payment_type==df_new.OLD_VALUE,how="inner").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   SortMergeJoin [payment_type#13218], [OLD_VALUE#13726], Inner
   :- ColumnarToRow
   :  +- PhotonResultStage
   :     +- PhotonSort [payment_type#13218 ASC NULLS FIRST]
   :        +- PhotonShuffleExchangeSource
   :           +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#10446]
   :              +- PhotonShuffleExchangeSink hashpartitioning(payment_type#13218, 7329)
   :                 +- PhotonScan parquet [vendor_id#13207,pickup_datetime#13208,dropoff_datetime#13209,passenger_count#13210,trip_distance#13211,pickup_longitude#13212,pickup_latitude#13213,rate_code_id#13214,store_and_fwd_flag#13215,dropoff_longitude#13216,dropoff_latitude#13217,payment_type#13218,fare_amount#13219,extra#13220,mta_tax#13221,tip_amount#13222,tolls_amount#13223,total_amount#13224] DataFilters: [isnotnull(payment_type#13218)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[dbfs:/databrick

In [0]:
from pyspark.sql.functions import broadcast
df_nytaxi.join(broadcast(df_new),on=df_nytaxi.payment_type==df_new.OLD_VALUE,how="inner").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   ColumnarToRow
   +- PhotonResultStage
      +- PhotonBroadcastHashJoin [payment_type#13218], [OLD_VALUE#13746], Inner, BuildRight, false, true
         :- PhotonScan parquet [vendor_id#13207,pickup_datetime#13208,dropoff_datetime#13209,passenger_count#13210,trip_distance#13211,pickup_longitude#13212,pickup_latitude#13213,rate_code_id#13214,store_and_fwd_flag#13215,dropoff_longitude#13216,dropoff_latitude#13217,payment_type#13218,fare_amount#13219,extra#13220,mta_tax#13221,tip_amount#13222,tolls_amount#13223,total_amount#13224] DataFilters: [isnotnull(payment_type#13218)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow], OptionalDataFilters: [hashedrelationcontains(payment_type#13218)], PartitionFilters: [], ReadSchema: struct<vendor_id:string,pickup_datetime:timestamp,dropoff_datetime:timestamp,passenger_count:int,..